# Project 3 — How little data and how few parameters does LoRA need on code-mixed sentiment?
**Dataset:** Spanglish SentiMix (SemEval-2020 Task 9; also the sentiment task in LinCE) — Spanish-English tweets, 3 classes.
**Model:** `xlm-roberta-base` + LoRA (Hu et al., 2021). **Baselines:** TF-IDF + logistic regression, zero-/few-shot `Qwen2.5-0.5B-Instruct`.
**Grid:** rank r in {8, 16} x training size n in {500, 2000}, 1 run each (+ optional head-only control, r = 0).

**How to use:** Runtime -> Change runtime type -> **T4 GPU**. Run the cells top to bottom. Results are saved to `/content/results.csv` and `/content/preds.pkl` after every run, so copy those to Drive if the session might die.

| Step | Cells | Rough time (T4, expectation, not measured) |
|---|---|---|
| 1. Data + fixed splits | 1-4 | 10-20 min (mostly getting the files) |
| 2. TF-IDF baseline | 5 | < 2 min |
| 3. Zero/few-shot LLM baseline | 6 | 5-10 min |
| 4. LoRA runs | 7-8 | a few min per run |
| 5. Results, plot, significance | 9-10 | < 2 min |
| 6. Error analysis | 11-12 | 30-45 min incl. hand-labelling |

## 1. Setup and configuration

In [ ]:
!pip -q install -U peft accelerate
# Colab ships an old torchao that the newest peft refuses to import; we don't use it, so remove it
!pip -q uninstall -y torchao

import os, re, time, pickle, random, warnings
import numpy as np, pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

SEED = 42
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "NO GPU - switch the runtime to T4")
assert device == "cuda", "No GPU found: Runtime -> Change runtime type -> T4 GPU (not TPU, not CPU), then reconnect and re-run from the top."

# ---- experiment configuration ----
LABELS = ["negative", "neutral", "positive"]      # fixed order, used everywhere
L2I = {l: i for i, l in enumerate(LABELS)}
N_TEST, N_VAL = 1500, 500                         # fixed held-out sets carved from train+dev
TRAIN_SIZES = [500, 2000]
RANKS = [8, 16]
EPOCHS = {500: 10, 2000: 6}
LR = 3e-4
INCLUDE_HEAD_ONLY = True                          # r=0 control: only the classifier head trains
DATA_DIR = "/content/data"                        # put the dataset files here (step 2)

## 2. Load the data (official SemEval-2020 Task 9 release on Zenodo)
The organisers' own release is a single 2.8 MB zip on Zenodo (record 3974927) containing both the Hinglish and Spanglish corpora. The cell below downloads it, unzips it, prints every file it finds, and automatically picks the **Spanglish train and dev** files by looking at their *contents* (Spanish tags, no Hindi tags), so it doesn't depend on guessing file names. Trial and test files are skipped; we carve our own test set in the next section.

You should see roughly **12,000 train + 3,000 dev** tweets. If the download fails (some networks block it), download `Semeval_2020_task9_data.zip` in your browser from zenodo.org/records/3974927, upload it to `/content/` as `semeval.zip`, and re-run. To use your own files instead, set `SOURCE = "local"` and put them in `/content/data/`.

In [ ]:
import urllib.request, zipfile, hashlib, glob

SOURCE = "zenodo"              # "zenodo" (official release) or "local" (your own files in DATA_DIR)
ZENODO_URLS = ["https://zenodo.org/records/3974927/files/Semeval_2020_task9_data.zip?download=1",
               "https://zenodo.org/record/3974927/files/Semeval_2020_task9_data.zip"]
ZENODO_MD5 = "4d93553c213b3a966b6d55b7a7c03be7"     # checksum published on the Zenodo record
ZIP_PATH, RAW_DIR = "/content/semeval.zip", "/content/semeval_raw"

LABEL_RE = re.compile(r"\b(positive|negative|neutral)\b", re.I)

def parse_conll(path):
    """CoNLL-style code-mixed file -> list of {tokens, tags, label}."""
    rows, toks, tags, label = [], [], [], None
    def flush():
        nonlocal toks, tags, label
        if toks and label:
            rows.append({"tokens": toks, "tags": tags, "label": label})
        toks, tags, label = [], [], None
    with open(path, encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line.strip():
                flush(); continue
            if line.startswith("meta") or line.startswith("#"):
                if toks: flush()
                m = LABEL_RE.search(line)
                label = m.group(1).lower() if m else None
                continue
            parts = line.split("\t")
            toks.append(parts[0]); tags.append(parts[1].lower() if len(parts) > 1 else "")
    flush()
    return rows

def detok(tokens):
    return re.sub(r"\s+([,.!?;:])", r"\1", " ".join(tokens))

def is_spanglish(rows):
    tagset = {t for r in rows[:300] for t in r["tags"]}
    return bool(tagset & {"es", "spa", "spn", "lang2"}) and not (tagset & {"hin", "hi"})

def fetch_zenodo():
    if not os.path.exists(ZIP_PATH):
        for url in ZENODO_URLS:
            try:
                urllib.request.urlretrieve(url, ZIP_PATH); break
            except Exception as e:
                print("  download failed:", url, "->", type(e).__name__, str(e)[:100])
    assert os.path.exists(ZIP_PATH), ("Could not download from Zenodo. Download the zip in your browser, "
                                      "upload it to /content/ as semeval.zip, then re-run this cell.")
    md5 = hashlib.md5(open(ZIP_PATH, "rb").read()).hexdigest()
    print("md5:", md5, "(matches the published checksum)" if md5 == ZENODO_MD5 else "(differs from the published checksum - continuing anyway)")
    with zipfile.ZipFile(ZIP_PATH) as z: z.extractall(RAW_DIR)
    for inner in glob.glob(RAW_DIR + "/**/*.zip", recursive=True):       # in case the zip contains zips
        with zipfile.ZipFile(inner) as z: z.extractall(os.path.dirname(inner))
    return RAW_DIR

root = fetch_zenodo() if SOURCE == "zenodo" else DATA_DIR
os.makedirs(root, exist_ok=True)
files = sorted(p for p in glob.glob(root + "/**/*", recursive=True) if os.path.isfile(p))
print(f"\n{len(files)} files under {root}:")
for p in files[:60]: print("  ", os.path.relpath(p, root))

frames = []
for p in files:
    name = os.path.basename(p).lower()
    if not any(k in name for k in ("train", "dev", "valid")) or any(k in name for k in ("test", "trial")):
        continue
    if name.endswith((".zip", ".pdf", ".png", ".json", ".md")):
        continue
    if name.endswith((".csv", ".tsv")):
        d = pd.read_csv(p, sep=None, engine="python")
        d["label"] = d["label"].astype(str).str.lower()
    else:
        rows = parse_conll(p)
        if not rows or not is_spanglish(rows):
            continue                                    # empty, or a Hinglish file
        d = pd.DataFrame(rows)
        d["text"] = d["tokens"].apply(detok)
    d["src"] = name; frames.append(d)
    print(f"using {os.path.relpath(p, root)}: {len(d)} tweets")

assert frames, ("No Spanglish train/dev files found. Look at the file list printed above and tell me what it shows "
                "(or set SOURCE = 'local' and upload train/dev files to /content/data/).")

pool = pd.concat(frames, ignore_index=True)
print("\nraw label values:", pool.label.value_counts().to_dict())
pool = pool[pool.label.isin(LABELS)]
pool = pool[pool.text.str.strip().str.len() > 0]
pool = pool.drop_duplicates("text").reset_index(drop=True)      # avoid train/test leakage via duplicates
pool["y"] = pool.label.map(L2I)

# code-mixing ratio per tweet = minority-language share of the language-tagged tokens (0 = monolingual, 0.5 = even mix)
L1 = {"lang1", "eng", "en"}; L2 = {"lang2", "spa", "spn", "es", "hin", "hi"}
def mix_ratio(tags):
    if not isinstance(tags, list): return np.nan
    n1 = sum(t in L1 for t in tags); n2 = sum(t in L2 for t in tags)
    return 0.0 if n1 + n2 == 0 else min(n1, n2) / (n1 + n2)
pool["mix"] = pool["tags"].apply(mix_ratio) if "tags" in pool else np.nan

print(pool.shape)
print(pool.label.value_counts())
display(pool[["text", "label", "mix"]].head(8))

**Sanity check before moving on:** the corpus papers report about 12,002 train and 2,998 dev tweets, with roughly 50% positive, 33% neutral and 17% negative. After de-duplication you should land a little under 15,000 in total. If the numbers or label values look very different, paste me the printed output and I will adjust.

## 3. Fixed splits (nested training sets)
One fixed **test** set and one fixed **validation** set are carved out first. Then the 2000-example training set is sampled from what is left, and the 500-example set is a stratified subset of it, so the two sizes are *nested* and the comparison is clean. The same splits are used by every method.

In [ ]:
rest, test = train_test_split(pool, test_size=N_TEST, stratify=pool.y, random_state=SEED)
rest, val  = train_test_split(rest, test_size=N_VAL,  stratify=rest.y,  random_state=SEED)
train2000, _ = train_test_split(rest, train_size=2000, stratify=rest.y, random_state=SEED)
train500,  _ = train_test_split(train2000, train_size=500, stratify=train2000.y, random_state=SEED)
assert set(train500.index) <= set(train2000.index)
assert not (set(test.index) & set(rest.index)) and not (set(val.index) & set(rest.index))

TRAIN = {500: train500, 2000: train2000}
y_test = test.y.values
for name, d in [("train500", train500), ("train2000", train2000), ("val", val), ("test", test), ("rest (all remaining)", rest)]:
    print(f"{name:22s} n={len(d):6d}", d.label.value_counts(normalize=True).round(3).to_dict())

## 4. Shared metric and logging helpers

In [ ]:
def evaluate(y_true, y_pred):
    f1s = f1_score(y_true, y_pred, labels=[0, 1, 2], average=None, zero_division=0)
    return {"macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
            "acc": accuracy_score(y_true, y_pred),
            **{f"f1_{l}": f for l, f in zip(LABELS, f1s)}}

RESULTS, PREDS = [], {}
def save_state():
    pd.DataFrame(RESULTS).to_csv("/content/results.csv", index=False)
    with open("/content/preds.pkl", "wb") as f: pickle.dump(PREDS, f)

def log_result(method, n, trainable_params, y_pred, **extra):
    r = {"method": method, "n_train": n, "trainable_params": trainable_params, **evaluate(y_test, y_pred), **extra}
    RESULTS.append(r); PREDS[(method, n)] = np.asarray(y_pred); save_state()
    print(f"{method:28s} n={n:<6d} macro-F1={r['macro_f1']:.3f}  acc={r['acc']:.3f}")
    return r

## 5. Baseline 1 — TF-IDF + logistic regression
Character n-grams (2-5) plus word n-grams (1-2): char n-grams matter for code-mixed, misspelled social-media text. `C` is tuned on the validation set for each training size so the baseline is not a straw man. The full-pool run (`n` = everything left) is a reference for how far more data would take a cheap model.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline, make_union
from sklearn.linear_model import LogisticRegression

def tfidf_lr(C):
    feats = make_union(
        TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True),
        TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=1, sublinear_tf=True))
    return make_pipeline(feats, LogisticRegression(C=C, max_iter=3000, class_weight="balanced"))

for n, df in {**TRAIN, len(rest): rest}.items():
    best = None
    for C in [1, 5, 20]:
        clf = tfidf_lr(C).fit(df.text, df.y)
        vf1 = f1_score(val.y, clf.predict(val.text), average="macro")
        if best is None or vf1 > best[0]: best = (vf1, C, clf)
    vf1, C, clf = best
    n_feat = sum(len(v.vocabulary_) for _, v in clf[0].transformer_list)
    log_result("TF-IDF+LR", n, n_feat * 3 + 3, clf.predict(test.text), val_macro_f1=vf1, C=C)

## 6. Baseline 2 — zero-shot and few-shot prompting (`Qwen2.5-0.5B-Instruct`)
Instead of parsing generated text, we read the model's next-token scores for the three label words and take the argmax. That removes formatting failures from the comparison. Few-shot examples (2 per class) come from the 500-example training set, never from the test set. For a stronger baseline, set `LLM_NAME` to `Qwen/Qwen2.5-1.5B-Instruct` (still fits a T4 in fp16).

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
LLM_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(LLM_NAME, padding_side="left")
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype=torch.float16).to(device).eval()

SYSTEM = ("You classify the sentiment of tweets that mix Spanish and English (Spanglish). "
          "Answer with exactly one word: positive, negative, or neutral.")

def build_prompt(text, shots=()):
    msgs = [{"role": "system", "content": SYSTEM}]
    for t, l in shots:
        msgs += [{"role": "user", "content": f"Tweet: {t}\nSentiment:"}, {"role": "assistant", "content": l}]
    msgs.append({"role": "user", "content": f"Tweet: {text}\nSentiment:"})
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

label_tok_ids = [tok.encode(l, add_special_tokens=False)[0] for l in LABELS]
assert len(set(label_tok_ids)) == 3, "label words share a first token - use a different label wording"

@torch.no_grad()
def llm_predict(texts, shots=(), bs=8):
    preds = []
    for i in range(0, len(texts), bs):
        prompts = [build_prompt(t, shots) for t in texts[i:i + bs]]
        enc = tok(prompts, return_tensors="pt", padding=True).to(device)
        h = llm.model(**enc).last_hidden_state[:, -1, :]          # last real token (left padding)
        logits = llm.lm_head(h)
        preds += logits[:, label_tok_ids].argmax(-1).cpu().tolist()
    return np.array(preds)

texts = test.text.tolist()
log_result(f"Zero-shot {LLM_NAME.split('/')[-1]}", 0, 0, llm_predict(texts))

shots = []
for l in LABELS:
    ex = train500[train500.label == l].sample(2, random_state=SEED)
    shots += list(zip(ex.text, ex.label))
random.Random(SEED).shuffle(shots)
log_result(f"6-shot {LLM_NAME.split('/')[-1]}", 6, 0, llm_predict(texts, shots))

del llm; torch.cuda.empty_cache()

## 7. LoRA on XLM-R base
`xlm-roberta-base` (~278M parameters) with LoRA adapters on the attention **query** and **value** projections, `alpha = 2r`, dropout 0.1. PEFT's sequence-classification mode also keeps the classification head trainable, so we log two numbers:
* `lora_params` = adapter weights only (this is what changes with the rank);
* `trainable_params` = adapters + classification head (the honest total, used on the plot's x-axis).

The head alone is ~0.6M parameters, which is why the optional `r = 0` control exists: it shows how much of the score comes from LoRA rather than from just training the head on top of a frozen encoder.

The best epoch is chosen on the **validation** set (never on test), then the test set is scored once.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

ENC_NAME, MAXLEN = "xlm-roberta-base", 96
enc_tok = AutoTokenizer.from_pretrained(ENC_NAME)

def collate(batch):
    labels = torch.tensor([b["labels"] for b in batch])
    out = enc_tok.pad([{k: v for k, v in b.items() if k != "labels"} for b in batch], return_tensors="pt")
    out["labels"] = labels
    return out

def make_loader(df, bs, shuffle):
    enc = enc_tok(list(df.text), truncation=True, max_length=MAXLEN)
    items = [{"input_ids": enc["input_ids"][i], "attention_mask": enc["attention_mask"][i], "labels": int(y)}
             for i, y in enumerate(df.y.values)]
    return torch.utils.data.DataLoader(items, batch_size=bs, shuffle=shuffle, collate_fn=collate)

@torch.no_grad()
def predict(model, loader):
    model.eval(); out = []
    for b in loader:
        b = {k: v.to(device) for k, v in b.items() if k != "labels"}
        with torch.autocast("cuda", dtype=torch.float16):
            out.append(model(**b).logits.float().cpu())
    return torch.cat(out).argmax(-1).numpy()

def run_lora(r, n, epochs, lr=LR, bs=16, seed=SEED):
    set_seed(seed)
    base = AutoModelForSequenceClassification.from_pretrained(ENC_NAME, num_labels=3)
    if r > 0:
        cfg = LoraConfig(task_type=TaskType.SEQ_CLS, r=r, lora_alpha=2 * r, lora_dropout=0.1,
                         target_modules=["query", "value"])
        model = get_peft_model(base, cfg)
    else:                                              # head-only control
        for p in base.parameters(): p.requires_grad = False
        for p in base.classifier.parameters(): p.requires_grad = True
        model = base
    model.to(device)

    trainable = [p for p in model.parameters() if p.requires_grad]
    n_trainable = sum(p.numel() for p in trainable)
    n_lora = sum(p.numel() for nm, p in model.named_parameters() if p.requires_grad and "lora_" in nm)
    total = sum(p.numel() for p in model.parameters())
    name = f"LoRA r={r}" if r > 0 else "Head-only (r=0)"
    print(f"\n=== {name} | n={n} | trainable={n_trainable:,} (lora={n_lora:,}) of {total:,} ({100*n_trainable/total:.2f}%)")

    train_loader, val_loader = make_loader(TRAIN[n], bs, True), make_loader(val, 64, False)
    opt = torch.optim.AdamW(trainable, lr=lr, weight_decay=0.01)
    total_steps = epochs * len(train_loader); warm = max(1, int(0.1 * total_steps))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min((s + 1) / warm, max(0.0, (total_steps - s) / max(1, total_steps - warm))))
    scaler = torch.amp.GradScaler("cuda")

    best_f1, best_state, t0 = -1.0, None, time.time()
    for ep in range(epochs):
        model.train()
        for b in train_loader:
            b = {k: v.to(device) for k, v in b.items()}
            with torch.autocast("cuda", dtype=torch.float16):
                loss = model(**b).loss
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            scaler.step(opt); scaler.update(); sched.step()
        vf1 = f1_score(val.y.values, predict(model, val_loader), average="macro")
        print(f"  epoch {ep+1:2d}/{epochs}  loss={loss.item():.3f}  val macro-F1={vf1:.3f}")
        if vf1 > best_f1:
            best_f1 = vf1
            best_state = {k: v.detach().cpu().clone() for k, v in model.named_parameters() if v.requires_grad}

    model.load_state_dict(best_state, strict=False)        # restore the best-on-validation epoch
    pred = predict(model, make_loader(test, 64, False))
    log_result(name, n, n_trainable, pred, lora_params=n_lora, rank=r, epochs=epochs,
               val_macro_f1=best_f1, train_time_s=round(time.time() - t0))
    del model, base, opt; torch.cuda.empty_cache()

### Run the grid
Order: all runs at n = 500 first, then n = 2000. If a run looks stuck (val macro-F1 flat at ~0.2-0.3 for several epochs), the 500-example setting may need a higher learning rate (try `LR = 5e-4`) or more epochs; note any change in your README, and apply it to *all* runs.

In [ ]:
configs = [(r, n) for n in TRAIN_SIZES for r in RANKS]
if INCLUDE_HEAD_ONLY: configs += [(0, n) for n in TRAIN_SIZES]
for r, n in configs:
    run_lora(r, n, EPOCHS[n])

## 8. Results table

In [ ]:
res = pd.DataFrame(RESULTS)
cols = ["method", "n_train", "trainable_params", "lora_params", "macro_f1", "acc",
        "f1_negative", "f1_neutral", "f1_positive", "val_macro_f1"]
table = res[cols].copy()
table.iloc[:, 4:] = table.iloc[:, 4:].astype(float).round(3)
display(table.sort_values(["n_train", "macro_f1"], ascending=[True, False]).reset_index(drop=True))
table.to_csv("/content/results_table.csv", index=False)

## 9. Trainable parameters vs. performance
Dashed lines = TF-IDF at the same training size; grey dotted lines = zero-/few-shot LLM. A LoRA point above its dashed line means LoRA beats the cheap baseline at that data size.

In [ ]:
colors = {500: "tab:blue", 2000: "tab:orange"}
fig, ax = plt.subplots(figsize=(7.5, 4.8))
for n in TRAIN_SIZES:
    d = res[res.method.str.startswith(("LoRA", "Head")) & (res.n_train == n)].sort_values("trainable_params")
    ax.plot(d.trainable_params, d.macro_f1, marker="o", color=colors[n], label=f"XLM-R + LoRA, n={n}")
    tf = res[(res.method == "TF-IDF+LR") & (res.n_train == n)].macro_f1.iloc[0]
    ax.axhline(tf, ls="--", color=colors[n], alpha=.7, label=f"TF-IDF+LR, n={n}")
for _, r in res[res.method.str.contains("shot")].iterrows():
    ax.axhline(r.macro_f1, ls=":", color="gray", alpha=.8)
    ax.text(ax.get_xlim()[0], r.macro_f1, f" {r.method}", va="bottom", fontsize=7, color="gray")
ax.set_xscale("log"); ax.set_xlabel("trainable parameters (log scale, adapters + head)")
ax.set_ylabel("macro-F1 on fixed test set"); ax.set_title("Spanglish SentiMix: parameters vs. macro-F1")
ax.legend(fontsize=7); ax.grid(alpha=.3); plt.tight_layout()
plt.savefig("/content/params_vs_f1.png", dpi=200); plt.show()

## 10. Does LoRA actually beat the baselines? (paired bootstrap)
With one training run per cell, small gaps can be noise. The paired bootstrap below resamples the **test set** and reports a 95% interval for the macro-F1 difference (LoRA minus baseline). This captures test-set noise only, not seed-to-seed training variance, so treat differences under ~2 points as inconclusive and say so in the write-up. If time allows, re-run the n = 500 cells with 2 more seeds.

In [ ]:
def paired_bootstrap(y, p1, p2, B=1000, seed=0):
    rng = np.random.default_rng(seed); diffs = []
    for _ in range(B):
        idx = rng.integers(0, len(y), len(y))
        diffs.append(f1_score(y[idx], p1[idx], average="macro") - f1_score(y[idx], p2[idx], average="macro"))
    return np.percentile(diffs, [2.5, 50, 97.5])

rows = []
for n in TRAIN_SIZES:
    tf_pred = PREDS[("TF-IDF+LR", n)]
    for method in [m for m in res.method.unique() if m.startswith(("LoRA", "Head"))]:
        lo, med, hi = paired_bootstrap(y_test, PREDS[(method, n)], tf_pred)
        verdict = "LoRA better" if lo > 0 else ("TF-IDF better" if hi < 0 else "inconclusive")
        rows.append({"n_train": n, "method": method, "diff_vs_TFIDF": round(med, 3),
                     "95% CI": f"[{lo:.3f}, {hi:.3f}]", "verdict": verdict})
display(pd.DataFrame(rows))

## 11. Error analysis on code-mixed text
1. **Confusion matrices** for the best LoRA run (chosen by *validation* macro-F1) and TF-IDF at the same n.
2. **Performance by degree of code-mixing**: tweets bucketed by the minority-language share of tagged tokens.
3. A **sample of 50 errors** exported to CSV for you to label by hand.

In [ ]:
lora_runs = res[res.method.str.startswith("LoRA")].sort_values("val_macro_f1")
best = lora_runs.iloc[-1]
best_n = int(best.n_train)
best_key, tf_key = (best.method, best_n), ("TF-IDF+LR", best_n)
p_lora, p_tf = PREDS[best_key], PREDS[tf_key]
print("Best LoRA run by validation macro-F1:", best_key)

for nm, p in [("LoRA", p_lora), ("TF-IDF", p_tf)]:
    cm = pd.DataFrame(confusion_matrix(y_test, p, labels=[0, 1, 2]),
                      index=[f"true {l}" for l in LABELS], columns=[f"pred {l}" for l in LABELS])
    print(f"\n{nm} confusion matrix"); display(cm)

test_ = test.copy()
test_["bucket"] = pd.cut(test_["mix"], [-0.01, 0.05, 0.25, 0.51],
                         labels=["near-monolingual", "light mixing", "heavy mixing"])
rows = []
for b in ["near-monolingual", "light mixing", "heavy mixing"]:
    m = (test_.bucket == b).values
    if m.sum() >= 30:
        rows.append({"bucket": b, "n_tweets": int(m.sum()),
                     "LoRA macro-F1": round(f1_score(y_test[m], p_lora[m], average="macro"), 3),
                     "TF-IDF macro-F1": round(f1_score(y_test[m], p_tf[m], average="macro"), 3)})
print("\nMacro-F1 by degree of code-mixing"); display(pd.DataFrame(rows))

In [ ]:
err = test_.assign(pred=[LABELS[i] for i in p_lora], tfidf_pred=[LABELS[i] for i in p_tf])
err = err[err.label != err.pred]
print(f"{len(err)} LoRA errors out of {len(test_)} test tweets")
sample = err.sample(min(50, len(err)), random_state=SEED)[["text", "label", "pred", "tfidf_pred", "mix"]].copy()
sample["error_type"] = ""        # <- fill in by hand
sample.to_csv("/content/error_analysis_to_label.csv", index=False)
display(sample.head(15))

**Label the 50 errors by hand** (open `/content/error_analysis_to_label.csv`, fill `error_type`). Suggested categories (add your own as you read):
* `code-switch at the sentiment word` — polarity carried by a word in the minority language
* `non-standard spelling / slang / Spanglish coinage`
* `negation` across languages (e.g. Spanish *no* + English adjective)
* `sarcasm or irony`
* `mixed polarity` (positive and negative in one tweet)
* `neutral vs. polar boundary` (most confusions usually sit here)
* `mentions / hashtags / URLs carry the content`
* `label looks wrong or ambiguous` (annotation noise)

Then count them: `pd.read_csv("/content/error_analysis_to_label.csv").error_type.value_counts()`.

## 12. Save everything

In [ ]:
print(pd.read_csv("/content/results_table.csv").to_string())
# from google.colab import files
# for f in ["results.csv", "results_table.csv", "params_vs_f1.png", "error_analysis_to_label.csv", "preds.pkl"]:
#     files.download("/content/" + f)

### README outline
1. **Question:** how little data and how few trainable parameters before LoRA beats TF-IDF and zero/few-shot prompting on Spanglish sentiment?
2. **Data:** SentiMix Spanglish (Patwa et al., 2020); splits, sizes, class balance, nested 500 ⊂ 2000.
3. **Methods:** TF-IDF+LR (C tuned on val), Qwen zero/6-shot, XLM-R + LoRA (r, alpha, targets, lr, epochs, best-on-val epoch).
4. **Results:** the table, the parameters-vs-F1 plot, the bootstrap verdicts.
5. **Error analysis:** confusion matrices, by-mixing-degree table, your hand-labelled error counts with 3-4 example tweets.
6. **Limitations:** single seed per cell, one train/test split, test noise only in the CI, small LLM baseline, tweets from one period/source.
7. **References:** Hu et al. (2021) LoRA; Patwa et al. (2020) SemEval-2020 Task 9; Aguilar et al. (2020) LinCE.